# PlantCLEF 2015 S-CNN(A) Genus Figures

Builds and displays draft-ready genus-level diagnostic plots for the trained VGG16 S-CNN(A) checkpoint.


## 1. Runtime Check


In [ ]:
import torch

print('CUDA:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')


## 2. Clone Or Update Project


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess

PROJECT_DIR = Path('/content/diploma')
REPO_URL = 'https://github.com/robodanill/diploma.git'
BRANCH = 'robodanill/main'


def clone_project():
    os.chdir('/content')
    if PROJECT_DIR.exists():
        shutil.rmtree(PROJECT_DIR)
    subprocess.run(['git', 'clone', '-b', BRANCH, REPO_URL, str(PROJECT_DIR)], check=True)


def pull_project() -> bool:
    if not (PROJECT_DIR / '.git').exists():
        return False
    result = subprocess.run(['git', 'pull', '--ff-only'], cwd=PROJECT_DIR)
    return result.returncode == 0


if PROJECT_DIR.exists():
    print(f'Trying to update existing project: {PROJECT_DIR}')
    if not pull_project():
        print('Pull failed or project is not a git repository; cloning a fresh copy.')
        clone_project()
else:
    print(f'Project not found at {PROJECT_DIR}; cloning a fresh copy.')
    clone_project()

os.chdir(PROJECT_DIR)
subprocess.run(['python', '-m', 'pip', 'install', '-e', '.[ml]'], check=True)

commit = subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], cwd=PROJECT_DIR, text=True).strip()
print(f'Project commit: {commit}')


## 3. Mount Google Drive


In [ ]:
from google.colab import drive

drive.mount('/content/drive')


## 4. Restore LeafScan Data And Paper60 Metadata


In [ ]:
%%bash
set -euo pipefail
trap 'echo "FAILED at line $LINENO: $BASH_COMMAND" >&2' ERR
export PYTHONUNBUFFERED=1
cd /content/diploma

ARCHIVE=/content/drive/MyDrive/PlantCLEF2015_leafscan_only.tar.gz
TEST_ARCHIVE=/content/drive/MyDrive/PlantCLEF2015_leafscan_test.tar.gz

echo "checking required archives"
ls -lh /content/drive/MyDrive/PlantCLEF2015*.tar.gz 2>/dev/null || true
if [ ! -f "$ARCHIVE" ]; then
  echo "Missing LeafScan training archive: $ARCHIVE" >&2
  exit 2
fi
if [ ! -f "$TEST_ARCHIVE" ]; then
  echo "Missing LeafScan test archive: $TEST_ARCHIVE" >&2
  exit 3
fi

rm -rf data/plantclef2015
mkdir -p data/plantclef2015

echo "extracting training archive: $ARCHIVE"
tar -xzf "$ARCHIVE" -C data/plantclef2015
test -f data/plantclef2015/leafscan/metadata.csv
cp data/plantclef2015/leafscan/metadata.csv data/plantclef2015/leafscan_metadata.csv

echo "extracting test archive: $TEST_ARCHIVE"
rm -rf data/plantclef2015/test_leafscan
mkdir -p data/plantclef2015/test_leafscan
tar -xzf "$TEST_ARCHIVE" -C data/plantclef2015/test_leafscan
test -f data/plantclef2015/test_leafscan/leafscan/metadata.csv
cp data/plantclef2015/test_leafscan/leafscan/metadata.csv data/plantclef2015/test_leafscan_metadata.csv

python - <<'PY2'
import csv
from collections import Counter, defaultdict
from pathlib import Path
from PIL import Image

with open('data/plantclef2015/leafscan_metadata.csv', newline='', encoding='utf-8') as file:
    source_rows = list(csv.DictReader(file))
with open('data/plantclef2015/test_leafscan_metadata.csv', newline='', encoding='utf-8') as file:
    test_rows = list(csv.DictReader(file))

test_species = {row['species'] for row in test_rows}
paper60_rows = [row for row in source_rows if row['species'] in test_species]
source_species = {row['species'] for row in source_rows}
missing_in_train = sorted(test_species - source_species)
if missing_in_train:
    raise RuntimeError(f'Missing test species in train metadata: {missing_in_train}')

fieldnames = list(source_rows[0].keys())
with open('data/plantclef2015/leafscan_paper60_metadata.csv', 'w', newline='', encoding='utf-8') as file:
    writer = csv.DictWriter(file, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(paper60_rows)

train_count_by_species = Counter(row['species'] for row in paper60_rows)
underfilled_species = sorted(species for species in test_species if train_count_by_species[species] < 6)
if underfilled_species:
    print('paper60 species with fewer than 6 train images:', underfilled_species)
    rows_by_species = defaultdict(list)
    for row in paper60_rows:
        rows_by_species[row['species']].append(row)
    augmented_dir = Path('data/plantclef2015/leafscan/augmented')
    augmented_dir.mkdir(parents=True, exist_ok=True)
    leafscan_root = Path('data/plantclef2015/leafscan')
    angles = [180, 90, 270, 15, -15]
    augmented_rows = []
    for species in underfilled_species:
        species_rows = rows_by_species[species]
        if not species_rows:
            raise RuntimeError(f'Cannot augment {species}: no train rows found')
        needed = 6 - len(species_rows)
        for index in range(needed):
            base_row = species_rows[index % len(species_rows)]
            source_path = Path(base_row['image_path'])
            if not source_path.is_absolute():
                source_path = leafscan_root / source_path
            angle = angles[index % len(angles)]
            output_name = f"{source_path.stem}_aug_rot{angle}_{index + 1}.jpg".replace('-', 'm')
            output_path = augmented_dir / output_name
            with Image.open(source_path) as image:
                image.convert('RGB').rotate(angle, expand=True, fillcolor=(255, 255, 255)).save(output_path, quality=95)
            augmented_row = dict(base_row)
            augmented_row['image_path'] = str(output_path.relative_to(leafscan_root))
            if 'source_xml' in augmented_row:
                augmented_row['source_xml'] = f"{augmented_row['source_xml']}#aug_rot{angle}"
            augmented_rows.append(augmented_row)
    paper60_rows.extend(augmented_rows)
    with open('data/plantclef2015/leafscan_paper60_metadata.csv', 'w', newline='', encoding='utf-8') as file:
        writer = csv.DictWriter(file, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(paper60_rows)
    print('paper60 augmented rows added:', len(augmented_rows))

train_count_by_species = Counter(row['species'] for row in paper60_rows)
underfilled_species = sorted(species for species in test_species if train_count_by_species[species] < 6)
if underfilled_species:
    raise RuntimeError(f'Cannot build 6-shot paper subset: {underfilled_species}')

print('leafscan source rows:', len(source_rows))
print('paper60 train rows:', len(paper60_rows))
print('paper60 train genera:', len({row['genus'] for row in paper60_rows}))
print('paper60 train species:', len({row['species'] for row in paper60_rows}))
print('paper60 test rows:', len(test_rows))
print('paper60 test genera:', len({row['genus'] for row in test_rows}))
print('paper60 test species:', len(test_species))
print('paper60 six-shot training rows:', 6 * len(test_species))
PY2


## 5. Locate The Genus Checkpoint On Drive


In [ ]:
from pathlib import Path
import shutil as shutil_module

MANUAL_GENUS_CHECKPOINT = ''
DRIVE_CHECKPOINT_ROOT = Path('/content/drive/MyDrive/diploma_checkpoints')
LOCAL_CHECKPOINT = Path('/content/diploma/checkpoints/scnn_genus_vgg16_best.pt')

if MANUAL_GENUS_CHECKPOINT:
    checkpoint = Path(MANUAL_GENUS_CHECKPOINT)
else:
    patterns = [
        'leafscan_vgg16/**/scnn_genus_vgg16_best.pt',
        '**/scnn_genus_vgg16_best.pt',
        'leafscan_vgg16/**/scnn_genus_vgg16.pt',
        '**/scnn_genus_vgg16.pt',
    ]
    candidates = []
    for pattern in patterns:
        candidates.extend(DRIVE_CHECKPOINT_ROOT.glob(pattern))
    candidates = sorted(set(candidates), key=lambda path: path.stat().st_mtime, reverse=True)
    if not candidates:
        raise FileNotFoundError(f'No VGG16 genus checkpoint found under {DRIVE_CHECKPOINT_ROOT}')
    checkpoint = candidates[0]

if not checkpoint.exists():
    raise FileNotFoundError(checkpoint)

LOCAL_CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)
shutil_module.copy2(checkpoint, LOCAL_CHECKPOINT)
Path('/content/diploma/.genus_checkpoint_path').write_text(str(LOCAL_CHECKPOINT), encoding='utf-8')
print('Selected Drive checkpoint:', checkpoint)
print('Copied to:', LOCAL_CHECKPOINT)
print('Size:', LOCAL_CHECKPOINT.stat().st_size)


## 6. Save Genus Draft Figures


In [ ]:
%%bash
set -euo pipefail
export PYTHONUNBUFFERED=1
cd /content/diploma

CHECKPOINT="$(cat .genus_checkpoint_path)"
BASE_OUT="/content/drive/MyDrive/diploma_diagnostics"
STAMP="$(date -u +%Y%m%dT%H%M%SZ)"

for MODE in comparator l1; do
  OUT_DIR="$BASE_OUT/genus_vgg16_${MODE}_${STAMP}"
  mkdir -p "$OUT_DIR"
  echo "=== genus artifacts score_mode=$MODE ==="
  python -u -m plant_classifier.training.eval_genus_cli \
    --config configs/leafscan_paper60_training.yaml \
    --query-config configs/leafscan_test.yaml \
    --checkpoint "$CHECKPOINT" \
    --max-species 0 \
    --references-per-genus 6 \
    --reference-level genus \
    --reference-seed 42 \
    --reference-split train \
    --score-mode "$MODE" \
    --output-dir "$OUT_DIR" \
    --top-k 5 15 30 50
  ls -lh "$OUT_DIR"
done


## 7. Display Genus Draft Figures Inline


In [ ]:
from pathlib import Path
from IPython.display import Image, Markdown, display

BASE = Path('/content/drive/MyDrive/diploma_diagnostics')


def latest_dir(pattern: str) -> Path | None:
    candidates = sorted(BASE.glob(pattern), key=lambda path: path.stat().st_mtime, reverse=True)
    return candidates[0] if candidates else None


def display_png(path: Path, width: int = 900) -> None:
    if not path.exists():
        display(Markdown(f'`{path}` not found'))
        return
    display(Markdown(f'`{path}`'))
    display(Image(filename=str(path), width=width))

for title, pattern in [
    ('Genus comparator artifacts', 'genus_vgg16_comparator_*'),
    ('Genus L1 artifacts', 'genus_vgg16_l1_*'),
]:
    display(Markdown(f'### {title}'))
    directory = latest_dir(pattern)
    if directory is None:
        display(Markdown('No artifact directory found yet.'))
        continue
    display(Markdown(f'Directory: `{directory}`'))
    for filename in [
        'topk_genus_accuracy.png',
        'genus_confusion_matrix.png',
        'genus_rank_histogram.png',
        'per_genus_top1_accuracy.png',
    ]:
        display_png(directory / filename)
